In [18]:
import pandas as pd
from customer_cleaner import CustomerCleaner

In [19]:
old_df=pd.read_csv("customer_churn.csv")

18.Load and clean the data using CustomerCleaner 

In [20]:
cleaner=CustomerCleaner(old_df)
df=cleaner.clean()
df

2026-08-07 10:26:10,974 - INFO - CustomerCleaner initialized.
2026-08-07 10:26:10,976 - INFO - Input DataFrame Shape: (7043, 21)
2026-08-07 10:26:10,978 - INFO - Starting data cleaning pipeline...
2026-08-07 10:26:10,979 - INFO - Standardizing column names...
2026-08-07 10:26:10,983 - INFO - Column names converted to snake_case.
2026-08-07 10:26:10,986 - INFO - Cleaning total_charges column...
2026-08-07 10:26:10,993 - INFO - Blank strings found: 0
2026-08-07 10:26:11,005 - INFO - Conversion completed. Total null values in total_charges: 11
2026-08-07 10:26:11,007 - INFO - Normalizing binary columns...
2026-08-07 10:26:11,009 - INFO - Processing column: partner
2026-08-07 10:26:11,013 - INFO - Processing column: dependents
2026-08-07 10:26:11,018 - INFO - Processing column: phone_service
2026-08-07 10:26:11,025 - INFO - Processing column: paperless_billing
2026-08-07 10:26:11,031 - INFO - Processing column: churn
2026-08-07 10:26:11,036 - INFO - Binary columns normalized.
2026-08-07 10

,customer_i_d,gender,senior_citizen,partner,dependents,tenure,phone_service,multiple_lines,internet_service,online_security,...,device_protection,tech_support,streaming_t_v,streaming_movies,contract,paperless_billing,payment_method,monthly_charges,total_charges,churn
0,7590-VHVEG,Female,0,1,0,1,0,No phone service,DSL,No,...,No,No,No,No,Month-to-month,1,Electronic check,29.85,29.85,0
1,5575-GNVDE,Male,0,0,0,34,1,No,DSL,Yes,...,Yes,No,No,No,One year,0,Mailed check,56.95,1889.50,0
2,3668-QPYBK,Male,0,0,0,2,1,No,DSL,Yes,...,No,No,No,No,Month-to-month,1,Mailed check,53.85,108.15,1
3,7795-CFOCW,Male,0,0,0,45,0,No phone service,DSL,Yes,...,Yes,Yes,No,No,One year,0,Bank transfer (automatic),42.30,1840.75,0
4,9237-HQITU,Female,0,0,0,2,1,No,Fiber optic,No,...,No,No,No,No,Month-to-month,1,Electronic check,70.70,151.65,1
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
7038,6840-RESVB,Male,0,1,1,24,1,Yes,DSL,Yes,...,Yes,Yes,Yes,Yes,One year,1,Mailed check,84.80,1990.50,0
7039,2234-XADUH,Female,0,1,1,72,1,Yes,Fiber optic,No,...,Yes,No,Yes,Yes,One year,1,Credit card (automatic),103.20,7362.90,0
7040,4801-JZAZL,Female,0,1,1,11,0,No phone service,DSL,Yes,...,No,No,No,No,Month-to-month,1,Electronic check,29.60,346.45,0
7041,8361-LTMKD,Male,1,1,0,4,1,Yes,Fiber optic,No,...,No,No,No,No,Month-to-month,1,Mailed check,74.40,306.60,1


19.Compute churn rate by contract type — sort descending. Which contract has the highest churn?

In [21]:
churn_rate= df.groupby("contract")["churn"].mean().sort_values(ascending=False)
churn_rate

contract
Month-to-month    0.427097
One year          0.112695
Two year          0.028319
Name: churn, dtype: float64

In [22]:
print(f"The Contract having highest churn is {churn_rate.index[0]} with a churn rate of {round(churn_rate.values[0],2)}") 

The Contract having highest churn is Month-to-month with a churn rate of 0.43


20.Compute churn rate by internet service type 

In [23]:
churn_rate_internet_service= df.groupby("internet_service")["churn"].mean().sort_values(ascending=False)
print("Churn Rate by Internet Service Type")
print(churn_rate_internet_service)

Churn Rate by Internet Service Type
internet_service
Fiber optic    0.418928
DSL            0.189591
No             0.074050
Name: churn, dtype: float64


21. Compute churn rate by payment method

In [26]:
churn_rate_payment_method=df.groupby("payment_method")["churn"].mean().sort_values(ascending=False)
print("The Churn rate by the Payment method is")
print(churn_rate_payment_method)

The Churn rate by the Payment method is
payment_method
Electronic check             0.452854
Mailed check                 0.191067
Bank transfer (automatic)    0.167098
Credit card (automatic)      0.152431
Name: churn, dtype: float64


22. Create a tenure_bucket column using pd.cut() with bins [0,12,24,48,72] and labels ["0-12","13-24","25-48","49-72"]. Compute churn rate per bucket. 

In [27]:
#Creating a Tenure Bucket Column with labels
df["tenure_bucket"]=pd.cut(
    df["tenure"],
    bins=[0,12,24,48,72],
    labels=["0-12","13-24","25-48","49-72"],
    include_lowest=True
)

In [31]:
churn_rate_tenure_bucket=df.groupby("tenure_bucket")["churn"].mean().sort_values(ascending=False)
print("Churn Rate by buckets:")
print(churn_rate_tenure_bucket)

Churn Rate by buckets:
tenure_bucket
0-12     0.474382
13-24    0.287109
25-48    0.203890
49-72    0.095132
Name: churn, dtype: float64


23. Compute average monthly_charges for churned vs non-churned customers 

In [36]:
avg_monthly_charges_churn_type=df.groupby("churn")["monthly_charges"].mean()
print("The average monthly charges for churned vs non-churned is")
print(avg_monthly_charges_churn_type)

The average monthly charges for churned vs non-churned is
churn
0    61.265124
1    74.441332
Name: monthly_charges, dtype: float64


24. Find the combination (contract + internet_service) with the highest churn rate using groupby on two columns 

In [35]:
churn_rate_comb_con_int_ser=df.groupby(["contract","internet_service"])["churn"].mean().sort_values(ascending=False)
print("The combination (contract + internet_service) with the highest churn rate is")
print(churn_rate_comb_con_int_ser)
print()
print()
print()
highest_combo = churn_rate_comb_con_int_ser.index[0]
highest_rate = churn_rate_comb_con_int_ser.iloc[0]

print(f"Contract: {highest_combo[0]}")
print(f"Internet Service: {highest_combo[1]}")
print(f"Churn Rate: {highest_rate:.2%}")

The combination (contract + internet_service) with the highest churn rate is
contract        internet_service
Month-to-month  Fiber optic         0.546053
                DSL                 0.322159
One year        Fiber optic         0.192950
Month-to-month  No                  0.188931
One year        DSL                 0.092982
Two year        Fiber optic         0.072261
One year        No                  0.024725
Two year        DSL                 0.019108
                No                  0.007837
Name: churn, dtype: float64



Contract: Month-to-month
Internet Service: Fiber optic
Churn Rate: 54.61%


25. Write a 3-sentence business summary: which segments are highest risk and what that means for the retention team 

# Business Summary

* Customers on **month-to-month** plan and customers using the **Fiber optic** internet service has the highest churn rate. This is the risk segment with a highest churn rate of approximately **54%**.

* The churn is highest among the customer within the **first 12 months**. The customers using **electronic check** and having **higher monthly charges** are most likely to leave.

* The retention team should prioritize **proactive outreach**, **targeted offers**, and **service improvements** for these high-risk segments to reduce customer churn and improve long-term retention.
